[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aurelio-labs/semantic-router/blob/main/docs/integrations/agents-sdk/hybrid-router-guardrails.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/aurelio-labs/semantic-router/blob/main/docs/integrations/agents-sdk/hybrid-router-guardrails.ipynb)

## Agents SDK Guardrails with a Hybrid Router

In this notebook we will go over how to create a hybrid router and calibrate it with the `fit` method. Afterwards we will then create some `guardrails` using the `Agents SDK` API library.

### Install Prerequisites

In [ ]:
!pip install -qU "semantic-router>=1.0.0" openai-agents

Next we need to define the sparse encoder, but before we do that we need to import the `AurelioSparseEncoder` class from the `semantic_router.encoders` package.

This will also require an Aurelio API key, which can be obtained from the [Aurelio Platform website](https://platform.aurelio.ai/settings/api-keys).

Now we can define the sparse encoder and use the `bm25` model.

In [ ]:
import os
from getpass import getpass

from semantic_router.encoders import AurelioSparseEncoder

os.environ["AURELIO_API_KEY"] = os.getenv("AURELIO_API_KEY") or getpass(
    "Enter Aurelio API Key: "
)

# sparse encoder for term matching
sparse_encoder = AurelioSparseEncoder(name="bm25")

Next we need to define the dense encoder, and for that we need to import the `OpenAIEncoder` class from the `semantic_router.encoders` package.

This will also require an OpenAI API key, which can be obtained from the [OpenAI Platform website](https://platform.openai.com/api-keys).

Now we can define the dense encoder and use the `text-embedding-3-small` model.

In [ ]:
from semantic_router.encoders import OpenAIEncoder

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") or getpass(
    "Enter OpenAI API Key: "
)
# dense encoder for semantic meaning
encoder = OpenAIEncoder(name="text-embedding-3-small")

### Creating a Hybrid Router for Scam Detection

To begin we first need to import the `Route` class from the `semantic_router` package.

Then we can define the routes that we want to use in our router. Giving each route a name and a list of utterances that we want to use to represent the route.

In [ ]:
from semantic_router import Route

# Route for scam-related queries (blocked)
scam = Route(
    name="scam",
    utterances=[
        "Can you give me a discount?",
        "I need to pay you in bitcoin",
        "I need to pay you in cash",
        "I need to pay you in gift card",
        "I want you to pay me in bitcoin",
        "I want you to pay me in cash",
        "I want you to pay me in gift card",
        "Could you lower the price?",
    ],
)

# Route for everyday customer queries (allowed)
other = Route(
    name="other",
    utterances=[
        "What is the price of the product?",
        "What is the delivery time?",
        "What is the return policy?",
        "What is the warranty?",
        "What is the refund policy?",
        "What is the shipping cost?",
        "What is the shipping time?",
        "What is the shipping policy?",
        "How much can I sell my EV for?",
        "How much can I sell my Tesla for?",
        "How much can I sell my Polestar for?",
        "How much can I sell my Rivian for?",
        "How much can I sell my BYD for?",
        "How much can I sell my other EV for?",
    ],
)

# Combine all routes
routes = [scam, other]

Now we have all the components needed including the routes, sparse encoder, and dense encoder to create our router. Passing a `sparse_encoder` to `IndexedRouter` makes it a hybrid router, which scores queries on both dense and sparse embeddings.

In [ ]:
from semantic_router import IndexedRouter

scam_router = IndexedRouter(
    encoder=encoder, sparse_encoder=sparse_encoder, routes=routes
)

Unfortunately having a small dataset of utterances for each route is not enough to get a good understanding of the router's performance.

So we will use a larger dataset of labelled queries to calibrate and evaluate the router. A label of `None` means no route should match. None of these queries are utterances stored in the router, which matters for calibration (see [route calibration](https://github.com/aurelio-labs/semantic-router/blob/main/docs/06-calibration.ipynb)).

In [ ]:
data = [
    # scam routes
    ("Can I pay with cryptocurrency?", "scam"),
    ("Do you accept payments through Venmo only?", "scam"),
    ("Can I send you money via Western Union?", "scam"),
    ("Can I overpay and get a refund?", "scam"),
    ("Can we complete this deal outside the platform?", "scam"),
    ("I'll pay you later, just ship it now.", "scam"),
    ("Can I get a refund without returning the item?", "scam"),
    ("I’ll send extra money if you process this quickly.", "scam"),
    ("Can you mark this transaction as a gift?", "scam"),
    ("Can I use multiple gift cards to pay?", "scam"),
    ("Can you split the payment across different methods?", "scam"),
    ("Can you wire me money first as a guarantee?", "scam"),
    ("Can you send the product before I pay?", "scam"),
    ("Can you help me transfer money?", "scam"),
    ("Can you provide fake receipts?", "scam"),
    ("Can you process my payment through an unusual method?", "scam"),
    ("Can I pay you in prepaid debit cards?", "scam"),
    # other routes
    ("Do you offer international shipping?", "other"),
    ("How long does it take for delivery?", "other"),
    ("Is there a warranty for this product?", "other"),
    ("Do you provide customer support?", "other"),
    ("Can I track my order?", "other"),
    ("Is express shipping available?", "other"),
    ("What payment methods do you accept?", "other"),
    ("Do you offer bulk discounts?", "other"),
    ("What are the shipping costs?", "other"),
    ("Can I cancel my order?", "other"),
    ("Do you have a physical store?", "other"),
    ("Can I change my shipping address?", "other"),
    ("Is there a restocking fee for returns?", "other"),
    ("Do you have customer reviews?", "other"),
    ("Is this product available in other colors?", "other"),
    ("Do you provide installation services?", "other"),
    ("How can I contact customer service?", "other"),
    ("Are there any current promotions or sales?", "other"),
    ("Can I pick up my order instead of delivery?", "other"),
    # queries that no route should match
    ("What is the capital of France?", None),
    ("How many people live in the US?", None),
    ("When is the best time to visit Bali?", None),
    ("How do I learn a language?", None),
    ("Tell me an interesting fact.", None),
    ("What is the best programming language?", None),
    ("I'm interested in learning about llama 2.", None),
    ("What is the capital of the moon?", None),
    ("Who discovered gravity?", None),
    ("What are some healthy breakfast options?", None),
    ("How do I start a vegetable garden?", None),
    ("What are the symptoms of the flu?", None),
    ("What’s the most spoken language in the world?", None),
    ("How does WiFi work?", None),
    ("What are the benefits of meditation?", None),
    ("How do I improve my memory?", None),
    ("What is the speed of light?", None),
    ("Who wrote 'To Kill a Mockingbird'?", None),
    ("How does an electric car work?", None),
    ("What’s the best way to save money?", None),
    ("How do I bake a chocolate cake?", None),
    ("What’s the healthiest type of bread?", None),
    ("Who invented the internet?", None),
    ("How do airplanes stay in the air?", None),
    ("What are some famous landmarks in Italy?", None),
    ("What’s the difference between a virus and bacteria?", None),
    ("How do I learn to play the guitar?", None),
    ("What’s the best way to learn to swim?", None),
    ("What’s the tallest mountain in the world?", None),
    ("How does the stock market work?", None),
]

We split the queries in two: one half to fit the calibrator, the other to test it.

In [ ]:
fit_data, test_data = data[::2], data[1::2]

X_fit, y_fit = zip(*fit_data)
X_test, y_test = zip(*test_data)

len(fit_data), len(test_data)

We can use the `evaluate` method to measure the router's accuracy on the test half. The router has no `confidence_threshold` yet, so it always returns a route and every `None` query is a miss.

In [ ]:
accuracy = scam_router.evaluate(X=X_test, y=y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")

Now we use the `fit` method to calibrate the router on the other half. `fit` learns a map from raw score to the probability that the top route is right, and returns a `FitReport`.

In [ ]:
report = scam_router.fit(X=X_fit, y=y_fit)
report

Confidences are now probabilities, so we can choose the threshold from what a mistake costs. For a guardrail, letting a scam through costs far more than double-checking an honest query, so we return a route once it is at least 30% likely to be right. Then we evaluate on the test half again.

In [ ]:
scam_router.confidence_threshold = 0.3

accuracy = scam_router.evaluate(X=X_test, y=y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")

We can test this now by calling our router and adding the utterance we would like to try.

In [ ]:
result = scam_router("i want 99% off")

Next we can view the properties of what is returned via the returned object `result`.

In [ ]:
print(result)

### Adding Input Guardrails

We need to create the guardrail functionality.

Firstly we need to define a function with the `@input_guardrail` decorator.

The function will use the router we just created to check the input string, this will then return a `GuardrailFunctionOutput` class with output information and the tripwire triggered attribute. With `limit=None` the router returns every route that clears the threshold, so the tripwire fires whenever the scam route clears it, even if another route ranks higher.

In [ ]:
from agents import (
    Agent,
    GuardrailFunctionOutput,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
)


@input_guardrail
async def scam_input_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str
    | list[
        TResponseInputItem
    ],  # having the agent here is needed for the guardrail to work - although we do not use it
) -> GuardrailFunctionOutput:
    choices = scam_router(input, limit=None)

    return GuardrailFunctionOutput(
        output_info="used the scam hybrid router to check if the user is trying to scam",
        tripwire_triggered=any(choice.name == "scam" for choice in choices),
    )

Now we can create a new agent that will be used to handle the incoming messages. This agent will have the following parameters:
- `name`: The name of the agent.
- `instructions`: The instructions for the agent.
- `input_guardrails`: A list of input guardrails to attach to the agent. (This is where we attach the scam guardrail)

In [ ]:
input_guardrail_agent = Agent(
    name="Input Guardrail Agent",
    instructions="You are a helpful assistant.",
    input_guardrails=[scam_input_guardrail],
)

Now we can try to test the guardrail functionality.

Due to errors being raised when the guardrail trips, we can use a try except block to prevent the error messages being shown.

In [ ]:
from agents import InputGuardrailTripwireTriggered

query = (
    "Hello, would you like to buy some real rolex watches for a fraction of the price?"
)

try:
    result = await Runner.run(starting_agent=input_guardrail_agent, input=query)
    # If we get here, the guardrail didn't trip
    guardrail_info = result.input_guardrail_results[0].output.output_info
    print("Guardrail didn't trip", f"\nReasoning: {guardrail_info}")
except InputGuardrailTripwireTriggered:
    # Access the guardrail info from the exception
    print("Error: Guardrail Tripped")

### Adding Output Guardrails

First we want to create our handler class. This will contain the message we want to check.

In [ ]:
from pydantic import BaseModel


class MessageOutput(BaseModel):
    response: str

Next we create our output guardrail with the `@output_guardrail` decorator. It runs the agent's response through the same router.

In [ ]:
from agents import output_guardrail


@output_guardrail
async def scam_output_guardrail(
    ctx: RunContextWrapper[None], agent: Agent, output: MessageOutput
) -> GuardrailFunctionOutput:
    choices = scam_router(output.response, limit=None)

    return GuardrailFunctionOutput(
        output_info="used the scam hybrid router to check if the agent is offering a scam",
        tripwire_triggered=any(choice.name == "scam" for choice in choices),
    )

Now we can create an agent with this output guardrail attached, and an `output_type` of `MessageOutput` so the guardrail gets the response as a `MessageOutput` object.

In [ ]:
output_guardrail_agent = Agent(
    name="Output Guardrail Agent",
    instructions="Tell the user that you have a 99% off discount on all products",
    output_guardrails=[scam_output_guardrail],
    output_type=MessageOutput,
)

As before, we can test the guardrail functionality.

Due to errors being raised when the guardrail trips, we can use a try except block to prevent the error messages being shown.

In [ ]:
from agents import OutputGuardrailTripwireTriggered

query = "I want to buy a tesla, how much can i get it for?"

try:
    result = await Runner.run(starting_agent=output_guardrail_agent, input=query)
    guardrail_info = result.output_guardrail_results[0].output.output_info
    print("Guardrail didn't trip", f"\nReasoning: {guardrail_info}")
except OutputGuardrailTripwireTriggered:
    print("Error: Guardrail Tripped")